# Claude Sonnet/Haiku V3 — correzione dei blocchi thinking/signature

**Questo notebook è disattivato per qualsiasi chiamata API finché non si modifica esplicitamente la cella pilota.** Non cancella né riscrive risultati V1/V2.

Fatto osservato: la prima A0 Sonnet V2 ha completato 7 chiamate LLM e 13 tool call, ma il campo `raw_response` include una rappresentazione Python del blocco Anthropic `thinking/signature` prima del testo JSON. Il validatore `json.loads` ha quindi dato `JSON non analizzabile`; NON è prova che il modello abbia generato testo diagnostico scorretto.

V3 introduce la sola estrazione dei blocchi `type=text` nelle finalizzazioni A0/A1/R8, Planning e PAVE, conservando i messaggi originali. Le chiamate Claude di V1/V2 non vengono ripetute sotto lo stesso `run_id`. V3 parte con nuovi identificativi create-only.


In [ ]:
import os, sys, json, subprocess, hashlib
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DATA_V3 = Path('/content/drive/MyDrive/Tesi_OpenRCA')
REPO_V3 = Path('/content/Tesi_OpenRCA_Claude_V3')
BRANCH_V3 = 'feature/claude-content-block-v3-2026-11-05'
REPO_URL = 'https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git'

if not REPO_V3.exists():
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH_V3,
                    REPO_URL,str(REPO_V3)],check=True)

def git_v3(*args):
    return subprocess.run(['git','-C',str(REPO_V3),*args],
        capture_output=True,text=True,check=True).stdout.strip()

assert git_v3('branch','--show-current') == BRANCH_V3
assert not git_v3('status','--porcelain'), 'STOP: checkout V3 sporco'
COMMIT_V3 = git_v3('rev-parse','HEAD')
print('Commit V3:',COMMIT_V3)
print('Nuovo checkout V3:',REPO_V3)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',
                str(REPO_V3/'requirements.txt')],check=True)

if str(REPO_V3/'src') not in sys.path:
    sys.path.insert(0,str(REPO_V3/'src'))
from evaluation import claude_development_campaign as cdc_v3
assert cdc_v3.PROTOCOL == 'openrca_claude_five_agents_development_v3'
assert cdc_v3.BRANCH == BRANCH_V3
print('V3 pronta: NESSUNA API chiamata.')

## 1. Verifica in sola lettura dei risultati precedenti

Confronta i SHA del piano V1, del piano V2 e dell'output già congelato V2. Non interpreta la predizione e non legge ground truth.

In [ ]:
V1 = 'openrca_claude_five_agents_development_v1'
V2 = 'openrca_claude_five_agents_development_v2'
V1_PLAN_SHA = 'fb828dfcd7f2ac9e07b6d916ffd51e61b6a23052c1edbf7a06e965acb5a26158'
V2_PLAN_SHA = '33c6f4f20f6d0462d215c83e6ae81c8344ba4332937801fae4ce98d5ba10cd7b'
V2_RESULT_SHA = 'fe639ca801d82111e23b0928b4dd48828f4533311a9439475e7e54b44eef6a1f'

def sha256_file(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

v1_plan = DATA_V3/'data/protocols/v2'/f'{V1}_plan_v1.json'
v2_plan = DATA_V3/'data/protocols/v2'/f'{V2}_plan_v1.json'
v1_run = V1+'__sonnet__cloudbed-1_q005__a0__r01'
v2_run = V2+'__sonnet__cloudbed-1_q005__a0__r01'
v1_base = DATA_V3/'data/results/v2'/V1
v2_base = DATA_V3/'data/results/v2'/V2
v1_claim = v1_base/(v1_run+'.claim.json')
v1_result = v1_base/(v1_run+'.json')
v2_result = v2_base/(v2_run+'.json')
v2_digest = v2_base/(v2_run+'.sha256.json')

assert sha256_file(v1_plan) == V1_PLAN_SHA
assert sha256_file(v2_plan) == V2_PLAN_SHA
assert v1_claim.exists() and not v1_result.exists()
assert sha256_file(v2_result) == V2_RESULT_SHA
assert json.loads(v2_digest.read_text(encoding='utf-8'))['output_sha256'] == V2_RESULT_SHA
print('V1 piano, V1 claim, V2 piano, V2 risultato: SHA/stato OK')
print('Nessuna modifica e nessuna API.')

## 2. Test offline di estrazione Anthropic e suite intera

Test sintetico con blocchi `thinking/signature` e `text`: il JSON finale deve essere leggibile e non contenere metadati. Vengono anche testate le normali funzioni del grafo e la conservazione degli artefatti V1/V2.

In [ ]:
env_offline = dict(os.environ)
env_offline['PYTHONPATH'] = str(REPO_V3/'src')
for k in ('ANTHROPIC_API_KEY','GEMINI_API_KEY','GOOGLE_API_KEY'):
    env_offline.pop(k,None)

for mask in ('test_anthropic_content_block_extraction.py','test_*.py'):
    print('ESECUZIONE TEST:',mask,flush=True)
    subprocess.run([
        sys.executable,'-m','unittest','discover',
        '-s',str(REPO_V3/'src/tests'),'-p',mask,'-v'
    ],cwd=REPO_V3,env=env_offline,check=True)

print('TUTTA LA SUITE V3: OK')
print('NESSUNA API, NESSUNO SCORING.')

## 3. Audit esplorativo del raw_response V2 — senza modificarlo

Il notebook controlla se il segmento testuale del vecchio `raw_response`, dopo la separazione dal primo blocco strutturato, è JSON sintatticamente corretto e conforme al contratto task-aware. Si tratta **soltanto di una diagnosi del parser**: il campo `prediction` originale della V2 resta `None`; un'eventuale conformità secondaria non diventa risultato V2 ufficiale. Non stampiamo firme, contenuti di thinking né la diagnosi.

In [ ]:
from output.v2_task_contract import audit_task_output
old = json.loads(v2_result.read_text(encoding='utf-8'))
raw = old.get('raw_response')
assert isinstance(raw,str)
assert old.get('prediction') is None
print('V2 raw_response lunghezza:',len(raw))
print('V2 json_status originale:',old.get('json_status'))
print('Contiene separatore tra blocchi:', '\n' in raw)

candidate = raw.split('\n',1)[-1] if '\n' in raw else ''
try:
    parsed = json.loads(candidate)
    print('Il testo successivo al primo blocco e JSON:',isinstance(parsed,dict))
    _,_,manifest = cdc_v3._load_gtfree(DATA_V3)
    case_id='cloudbed-1_q005'
    case=next(c for c in cdc_v3.build_plan(DATA_V3,COMMIT_V3)['cases']
              if c['case_id']==case_id)
    audit = audit_task_output(
        candidate,
        task_index=case['task_index'],
        instruction=manifest[case_id]['instruction'],
        start_unix=case['start_unix'], end_unix=case['end_unix'],
    )
    print('Contratto task-aware del testo ricostruito:',audit['valid'])
    print('Errore nel solo audit:',audit['error'])
except (ValueError,TypeError) as exc:
    print('Non dimostrata ricostruzione offline:',
          type(exc).__name__,str(exc)[:200])
print('V2 NON modificata, nessun output sostituito.')

## 4. Preview protocollo V3 — zero scritture

Il piano V3 parte dal medesimo preregistro GT-free Gemini V3, stessi otto casi e cinque varianti, ma con nuovi run ID. I risultati storici V1 e V2 non vengono riciclati.

In [ ]:
preview = cdc_v3.build_plan(DATA_V3,COMMIT_V3)
assert preview['run_count']==80
assert len(preview['runs'])==80
assert preview['response_text_extraction']=='text_blocks_only_v1'
assert preview['recovery_provenance']['v2_plan_sha256']==V2_PLAN_SHA
assert preview['recovery_provenance']['v2_infrastructure_output_sha256']==V2_RESULT_SHA
first=preview['runs'][0]
assert first['model_key']=='sonnet'
assert first['case_id']=='cloudbed-1_q005'
assert first['variant']=='A0'
print('Protocollo:',preview['protocol'])
print('Commit:',preview['git_commit'])
print('Run Sonnet:',sum(r['model_key']=='sonnet' for r in preview['runs']))
print('Run Haiku:',sum(r['model_key']=='haiku' for r in preview['runs']))
print('Primo nuovo identificativo:',first['run_id'])
print('PREVIEW SOLO OFFLINE — NESSUNA SCRITTURA.')

## 5. Piano V3 create-only — disattivato

Se le celle precedenti sono tutte riuscite, imposta `CREA_PIANO_V3=True` **una volta sola**. Questa è la sola scrittura su Drive di preparazione dell'esperimento. Non cambia V1/V2.

In [ ]:
CREA_PIANO_V3 = False  # Attivare SOLO dopo tutti i test offline

p3 = cdc_v3._plan_path(DATA_V3)
if CREA_PIANO_V3:
    if not p3.exists():
        print('Creato piano V3:',cdc_v3.write_plan(DATA_V3,REPO_V3))
    else:
        print('Piano V3 gia presente: nessuna sovrascrittura')
else:
    print('Piano V3 NON creato: default sicuro.')
if p3.exists():
    state3 = cdc_v3.preflight(DATA_V3,REPO_V3)
    print('Piano SHA-256:',state3['plan_sha256'])
    print('Sonnet:',state3['models']['sonnet'])
    print('Haiku:',state3['models']['haiku'])
print('Nessuna API chiamata.')

## 6. Secret Anthropic — nessuna chiamata API

Leggi la chiave da Colab Secrets. Non incollarla in chat o nel repository.

In [ ]:
from google.colab import userdata
secret = userdata.get('ANTHROPIC_API_KEY')
assert secret and secret.strip()
os.environ['ANTHROPIC_API_KEY'] = secret
del secret
print('ANTHROPIC_API_KEY disponibile (valore nascosto).')
print('Nessuna API chiamata.')

## 7. Una sola nuova A0 Sonnet V3 — OFF per default

**Non abilitare fino a verifica del piano e nuova autorizzazione esplicita del pilota.** Stesso caso `cloudbed-1_q005`, ma nuovo `run_id` V3; nessun retry del tentativo V1 o della run V2. Il budget in USD è indicativo, non hard cap. Se la chiamata viene interrotta, esegui soltanto l'audit successivo, MAI ripetere la stessa run.

In [ ]:
AVVIA_PILOTA_SONNET_V3 = False
BUDGET_RIFERIMENTO_USD = 0.50  # non un tetto tecnico

if not AVVIA_PILOTA_SONNET_V3:
    print('API Claude DISABILITATA: nessuna run avviata.')
else:
    state3 = cdc_v3.preflight(DATA_V3,REPO_V3)
    assert state3['models']['sonnet']=={'ready':40}
    assert state3['models']['haiku']=={'ready':40}
    first = state3['runs'][0]
    assert first['variant']=='A0' and first['case_id']=='cloudbed-1_q005'
    assert first['model_key']=='sonnet'
    print('Nuova run V3:',first['run_id'])
    result = cdc_v3.run_batch(
        DATA_V3,REPO_V3,model_key='sonnet',max_new_runs=1,
        confirm_api=True,approved_budget_usd=BUDGET_RIFERIMENTO_USD)
    print(result)

## 8. Audit sempre sicuro — solo lettura

Verifica integrità, contratto JSON e prestazioni; eseguilo dopo l'eventuale pilota. Non chiama Claude.

In [ ]:
p3 = cdc_v3._plan_path(DATA_V3)
if p3.exists():
    state3=cdc_v3.preflight(DATA_V3,REPO_V3)
    first=state3['runs'][0]
    claim,output,digest=cdc_v3._paths(DATA_V3,first['run_id'])
    print('Run:',first['run_id'])
    print('Stato:',first['status'])
    print('Claim/JSON/SHA:',claim.exists(),output.exists(),digest.exists())
    print('Sonnet:',state3['models']['sonnet'])
    print('Haiku:',state3['models']['haiku'])
    if output.exists():
        data=json.loads(output.read_text(encoding='utf-8'))
        print('json_status:',data.get('json_status'))
        print('contract valid:',(data.get('output_contract_v2') or {}).get('valid'))
        print('prediction presente:',data.get('prediction') is not None)
        print('errore:',data.get('error'))
        print('metriche:',data.get('metrics'))
        if digest.exists():
            sidecar=json.loads(digest.read_text(encoding='utf-8'))
            print('SHA output valido:',
                  sidecar['output_sha256']==cdc_v3._sha(output))
else:
    print('Piano V3 non presente, nessun esperimento avviato.')
print('SOLO LETTURA — nessuna API Claude.')